In [16]:
import polars as pl                      # основная библиотека для таблиц
import polars.selectors as cs            # «выбиралки» столбцов по типу (cs.numeric() и т.п.)
import numpy as np                       # массивы чисел, нужен для метрик
import lightgbm as lgb                   # градиентный бустинг (ансамбль деревьев)
from sklearn.metrics import roc_auc_score, average_precision_score

pl.Config.set_tbl_cols(-1)               # показывать ВСЕ столбцы таблиц (-1 = без ограничения)
pl.Config.set_tbl_rows(40)               # показывать до 40 строк

data_path = "../data/raw/"
df_clients = pl.read_csv(f"{data_path}clients.csv", try_parse_dates=True)          # справочник клиентов
df_monthly = pl.read_csv(f"{data_path}monthly_client.csv", try_parse_dates=True)   # главная таблица
df_nso     = pl.read_csv(f"{data_path}monthly_nso.csv", try_parse_dates=True)      # срочные вклады

df = (
    df_monthly
    .sort(["client_id", "report_date"])                  # порядок важен: сдвиги и скользящие окна
    .with_columns(
        pl.col("report_date").rank("dense").cast(pl.Int32).alias("month_idx")  # дата -> номер месяца 1..29
    )
)
max_idx = df["month_idx"].max()                          # последний месяц (29)
print(df.shape, "| клиентов:", df["client_id"].n_unique(), "| месяцев:", max_idx)


(87000, 70) | клиентов: 3000 | месяцев: 29


In [17]:
print(df.shape, "| клиентов:", df["client_id"].n_unique(), "| месяцев:", max_idx)
# Если строк меньше, чем клиентов × месяцев, кто-то «пропадает» из данных
print(df.group_by("client_id").len()["len"].value_counts().sort("len"))


(87000, 70) | клиентов: 3000 | месяцев: 29
shape: (1, 2)
┌─────┬───────┐
│ len ┆ count │
│ --- ┆ ---   │
│ u32 ┆ u32   │
╞═════╪═══════╡
│ 29  ┆ 3000  │
└─────┴───────┘


In [18]:
# 1. ОБЯЗАТЕЛЬНО: гарантируем правильный порядок строк (клиент -> хронология)
df = df.sort(["client_id", "month_idx"])

LIAB_DROP = 0.5   # пассивы упали ниже 50%
TXN_DROP  = 0.5   # транзакции упали ниже 50%

df = df.with_columns(
    # Пик по клиенту
    pl.col("total_liabilities_end").rolling_mean(3, min_samples=2)
      .cum_max().shift(1).over("client_id").alias("peak_liab"),
    pl.col("txn_cnt").cast(pl.Float64).rolling_mean(3, min_samples=2)
      .cum_max().shift(1).over("client_id").alias("peak_txn"),
).with_columns(
    # ИСПРАВЛЕНИЕ 1: используем "ИЛИ" (|) вместо "И" (&) — оттоком считается падение денег ИЛИ операций
    ((pl.col("total_liabilities_end") < LIAB_DROP * pl.col("peak_liab")) |
     (pl.col("txn_cnt") < TXN_DROP * pl.col("peak_txn"))).alias("is_low")
).with_columns(
    # Подтверждение падения
    (pl.col("is_low") & pl.col("is_low").shift(-1).over("client_id")).alias("is_low_confirmed")
).with_columns(
    pl.when(pl.col("is_low_confirmed")).then(pl.col("month_idx"))
      .min().over("client_id").alias("churn_idx_rule"),
    pl.col("month_idx").max().over("client_id").alias("last_idx"),
).with_columns(
    pl.min_horizontal(
        "churn_idx_rule",
        pl.when(pl.col("last_idx") < max_idx).then(pl.col("last_idx") + 1),
    ).alias("churn_idx")
).with_columns(
    pl.when(pl.col("month_idx") >= max_idx - 1).then(None)
      .when(pl.col("month_idx") + 1 == pl.col("churn_idx")).then(1)
      .otherwise(0)
      .cast(pl.Int8).alias("churn_next_month")
)

# Оставляем только строки с известным таргетом (убираем None) и до фактического ухода
df = df.filter(
    pl.col("churn_idx").is_null() | (pl.col("month_idx") < pl.col("churn_idx"))
)

print("Доля 1 по строкам (после исправления):", df["churn_next_month"].mean())

Доля 1 по строкам (после исправления): 0.015473458206855705


In [19]:
# НСО: агрегируем до уровня клиент + месяц
nso_agg = df_nso.group_by(["client_id", "report_date"]).agg(
    pl.col("early_close_flag").max().alias("nso_early_close_any"),
    pl.col("outflow_amt").sum().alias("nso_outflow_sum"),
)

df = (
    df.join(nso_agg, on=["client_id", "report_date"], how="left")
      .join(df_clients.select("client_id", "registration_date", "employee_cnt_start"),
            on="client_id", how="left")
      .sort(["client_id", "report_date"])          # join может сбить порядок, сортируем снова
      .with_columns(
          pl.col("nso_early_close_any", "nso_outflow_sum").fill_null(0),
          ((pl.col("report_date") - pl.col("registration_date")).dt.total_days() / 30.4)
              .alias("client_age_m"),
          pl.col("segment_name").replace_strict(
              {"MASS": 0, "Alfa Only": 1, "A-Club": 2}, default=-1),
      )
)

base_cols = ["total_liabilities_end", "demand_balance_avg", "nso_balance_avg", "txn_cnt",
             "txn_amt_in", "txn_amt_out", "active_days", "uniq_ca_cnt",
             "target_outflow_amt", "txn_oth_bank_amt", "salary_fund_amt"]

feat_exprs = []
for c in base_cols:
    x = pl.col(c).cast(pl.Float64)
    feat_exprs += [
        (x / (x.shift(1).over("client_id") + 1)).alias(f"{c}_chg_1m"),   # изменение к прошлому месяцу
        (x / (x.shift(3).over("client_id") + 1)).alias(f"{c}_chg_3m"),   # к 3 месяцам назад
        (x / (x.rolling_mean(6, min_samples=3).shift(1).over("client_id") + 1)).alias(f"{c}_vs_6m"),
        x.rolling_std(3, min_samples=2).over("client_id").alias(f"{c}_std_3m"),  # нестабильность
    ]

feat_exprs += [
    (pl.col("total_liabilities_end") / (pl.col("peak_liab") + 1)).alias("liab_vs_peak"),
    (pl.col("txn_cnt") / (pl.col("peak_txn") + 1)).alias("txn_vs_peak"),
    # сколько из последних 3 месяцев пассивы падали
    (pl.col("total_liabilities_end") < pl.col("total_liabilities_end").shift(1))
        .cast(pl.Int8).rolling_sum(3, min_samples=1).over("client_id").alias("liab_down_months_3m"),
]

df = df.with_columns(feat_exprs).with_columns(pl.col(pl.Boolean).cast(pl.Int8))


## Разбивка по времени

In [20]:
data = df.filter(pl.col("churn_next_month").is_not_null()).drop_nulls("peak_liab")

# при max_idx = 29: train ≤16 | зазор | valid 19–21 | зазор | test 24–27
train = data.filter(pl.col("month_idx") <= max_idx - 13)
valid = data.filter(pl.col("month_idx").is_between(max_idx - 10, max_idx - 8))
test  = data.filter(pl.col("month_idx").is_between(max_idx - 5,  max_idx - 2))

drop_cols = [
    "client_id", "report_date", "cpr_id", "month_idx", "registration_date",
    "churn_idx", "churn_idx_rule", "last_idx", "is_low_confirmed", 
    "is_low", "peak_liab", "peak_txn", "churn_next_month", 'target_outflow_amt'
]
features = [c for c in data.columns if c not in drop_cols and not c.startswith('target_outflow')]

for name, part in [("train", train), ("valid", valid), ("test", test)]:
    print(name, part.shape, "доля оттока:", round(part["churn_next_month"].mean(), 4))


train (37650, 130) доля оттока: 0.0153
valid (6766, 130) доля оттока: 0.0216
test (8114, 130) доля оттока: 0.0154


In [21]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score

X_tr, y_tr = train.select(features).to_pandas(), train["churn_next_month"].to_numpy()
X_va, y_va = valid.select(features).to_pandas(), valid["churn_next_month"].to_numpy()
X_te, y_te = test.select(features).to_pandas(),  test["churn_next_month"].to_numpy()

cat_features = ['segment_name', 'kp_id']
for col in cat_features:
    if col in X_tr.columns:
        X_tr[col] = X_tr[col].astype('category')
        X_va[col] = X_va[col].astype('category')
        X_te[col] = X_te[col].astype('category')

model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.03,
    num_leaves=31,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1,
)
model.fit(
    X_tr, y_tr,
    eval_set=[(X_va, y_va)],
    eval_metric="auc",  # AUC гораздо стабильнее для early stopping
    categorical_feature=cat_features,
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)],
)


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[50]	valid_0's auc: 0.861614	valid_0's binary_logloss: 0.0781724
Early stopping, best iteration is:
[47]	valid_0's auc: 0.862471	valid_0's binary_logloss: 0.0783554


,learning_rate,0.03
,n_estimators,1000
,min_child_samples,50
,subsample,0.8
,subsample_freq,1
,colsample_bytree,0.8
,random_state,42
,verbose,-1
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1


In [22]:
p_te = model.predict_proba(X_te)[:, 1]

print("ROC-AUC:", round(roc_auc_score(y_te, p_te), 3))            # 0.5 = монетка, 1.0 = идеал
print("PR-AUC :", round(average_precision_score(y_te, p_te), 3))  # честнее при редком оттоке
print("Наивный бейзлайн (только liab_vs_peak):",
      round(roc_auc_score(y_te, -X_te["liab_vs_peak"]), 3))

# Бизнес-метрика: менеджеры обзванивают топ-10% клиентов с самым высоким риском
scored = test.select("client_id", "report_date", "churn_next_month").with_columns(
    pl.Series("score", p_te))
top = scored.sort("score", descending=True).head(int(0.1 * scored.height))
print("Recall@10%   :", round(top["churn_next_month"].sum() / scored["churn_next_month"].sum(), 3))
print("Precision@10%:", round(top["churn_next_month"].mean(), 3))


ROC-AUC: 0.906
PR-AUC : 0.507
Наивный бейзлайн (только liab_vs_peak): 0.659
Recall@10%   : 0.72
Precision@10%: 0.111


In [23]:
imp = pl.DataFrame({
    "feature": features,
    "gain": model.booster_.feature_importance("gain"),
}).sort("gain", descending=True)
print(imp.head(20))


shape: (20, 2)
┌──────────────────────────────┬─────────────┐
│ feature                      ┆ gain        │
│ ---                          ┆ ---         │
│ str                          ┆ f64         │
╞══════════════════════════════╪═════════════╡
│ op7_cnt                      ┆ 5112.845545 │
│ txn_self_amt                 ┆ 2973.311503 │
│ op7_amt                      ┆ 2512.0636   │
│ txn_cnt                      ┆ 2271.235917 │
│ txn_vs_peak                  ┆ 1961.627558 │
│ txn_amt_in_vs_6m             ┆ 1498.345545 │
│ txn_cnt_out                  ┆ 1225.529088 │
│ nso_balance_avg_chg_1m       ┆ 901.486824  │
│ txn_cnt_vs_6m                ┆ 838.619273  │
│ txn_amt_in_chg_3m            ┆ 737.208866  │
│ txn_cnt_chg_1m               ┆ 682.850663  │
│ nso_balance_avg_chg_3m       ┆ 615.60246   │
│ liab_vs_peak                 ┆ 566.179539  │
│ demand_balance_avg_chg_3m    ┆ 552.311258  │
│ avg_rate                     ┆ 543.424605  │
│ uniq_ca_cnt_vs_6m            ┆ 485.746724  

In [24]:
# --- MVP: Формирование витрины удержания для CRM (на 31 мая 2026) ---

# 1. Берем самый актуальный срез данных (месяц 29 = 31 мая 2026)
latest_idx = df["month_idx"].max()
current_month_data = df.filter(pl.col("month_idx") == latest_idx)
X_current = current_month_data.select(features).to_pandas()

for col in cat_features:
    if col in X_current.columns:
        X_current[col] = X_current[col].astype("category")

# 2. Рассчитываем риск ухода для клиентов на актуальном срезе
current_risk = model.predict_proba(X_current)[:, 1]

scored_current = current_month_data.with_columns(
    pl.Series("risk_score", current_risk)
)

# 3. Точная расшифровка причин (убираем пустые заглушки по топ-факторам бустинга)
scored_current = scored_current.with_columns(
    pl.when(pl.col("nso_early_close_any") == 1)
      .then(pl.lit("Досрочно закрыт срочный депозит (НСО)"))
      .when(pl.col("txn_vs_peak") < 0.5)
      .then(pl.lit("Число операций упало более чем на 50% от пика"))
      .when(pl.col("liab_vs_peak") < 0.5)
      .then(pl.lit("Остатки на счетах просели более чем на 50% от пика"))
      .when(pl.col("txn_amt_in_vs_6m") < 0.6)
      .then(pl.lit("Поступления на 40%+ ниже среднего за полгода"))
      .when(pl.col("txn_cnt_chg_1m") < 0.6)
      .then(pl.lit("Резкое падение числа операций за последний месяц"))
      .when(pl.col("total_liabilities_end_chg_1m") < 0.6)
      .then(pl.lit("Снижение остатка средств более чем на 40% за месяц"))
      .otherwise(pl.lit("Затухание общей финансовой активности контрагента"))
      .alias("churn_reason")
)

# 4. Отбираем топ-10% самых рискованных клиентов и разделяем каналы коммуникации
cutoff = scored_current["risk_score"].quantile(0.90)

mvp_action_list = (
    scored_current
    .filter(pl.col("risk_score") >= cutoff)
    .select([
        "client_id",
        "report_date",
        pl.col("risk_score").round(3).alias("churn_probability"),
        pl.when(pl.col("risk_score") >= 0.5).then(pl.lit("КРИТИЧЕСКИЙ"))
          .otherwise(pl.lit("ВЫСОКИЙ")).alias("priority"),
        "churn_reason",
        # Решение проблемы выгорания менеджеров (разделение нагрузки):
        pl.when(pl.col("risk_score") >= 0.5)
          .then(pl.lit("Персональный звонок (аудит тарифа)"))
          .otherwise(pl.lit("Digital-пуш / спецпредложение в банк-клиенте"))
          .alias("recommended_action")
    ])
    .sort("churn_probability", descending=True)
)

# 5. Экспортируем витрину в CSV для CRM
mvp_action_list.write_csv("crm_churn_retention_list.csv")

print(f"Готово! Витрина MVP сформирована на дату: {scored_current['report_date'].max()}")
print(f"Всего клиентов в фокусе удержания: {mvp_action_list.height}")
print(mvp_action_list.head(10))

Готово! Витрина MVP сформирована на дату: 2026-05-31
Всего клиентов в фокусе удержания: 196
shape: (10, 6)
┌───────────┬─────────────┬───────────────────┬─────────────┬───────────────────┬──────────────────┐
│ client_id ┆ report_date ┆ churn_probability ┆ priority    ┆ churn_reason      ┆ recommended_acti │
│ ---       ┆ ---         ┆ ---               ┆ ---         ┆ ---               ┆ on               │
│ str       ┆ date        ┆ f64               ┆ str         ┆ str               ┆ ---              │
│           ┆             ┆                   ┆             ┆                   ┆ str              │
╞═══════════╪═════════════╪═══════════════════╪═════════════╪═══════════════════╪══════════════════╡
│ CL0002830 ┆ 2026-05-31  ┆ 0.577             ┆ КРИТИЧЕСКИЙ ┆ Число операций    ┆ Персональный     │
│           ┆             ┆                   ┆             ┆ упало более чем…  ┆ звонок (аудит    │
│           ┆             ┆                   ┆             ┆                   ┆ тар